# 1. Pipeline: folds, input conditions and features

Builds the outer folds, the four input conditions and the 512-d features for one dataset.
Everything that learns from data (U-Net, CNN head) is fitted inside each outer fold.

In [ ]:
import re
from pathlib import Path

import cv2
import joblib
import numpy as np
import pywt
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold

DATASET = "BUSI"  # or "BUS-UCLM"
DATA_DIR = Path("../data") / DATASET
OUT_DIR = Path("../outputs") / DATASET
(OUT_DIR / "features").mkdir(parents=True, exist_ok=True)

SIZE = 224
N_FOLDS = 5
SEED = 42
N_AUG = 5
CLASSES = ["benign", "malignant", "normal"]
CONDITIONS = ["raw", "predicted", "shuffled", "oracle"]

## Data and outer folds

BUSI has no patient identifiers, so every image is its own group. BUS-UCLM is grouped by patient.

In [ ]:
def read_busi():
    for label, cls in enumerate(CLASSES):
        folder = DATA_DIR / cls
        files = sorted(p.name for p in folder.glob("*.png"))
        for name in files:
            if "_mask" in name:
                continue
            img = cv2.imread(str(folder / name))
            mask = np.zeros(img.shape[:2], bool)
            for mask_name in files:
                if mask_name.startswith(name[:-4] + "_mask"):
                    m = cv2.imread(str(folder / mask_name), 0)
                    mask |= cv2.resize(m, img.shape[1::-1], interpolation=cv2.INTER_NEAREST) > 127
            yield img, mask, label, f"{cls}/{name}"


def read_uclm():
    for path in sorted((DATA_DIR / "images").glob("*.png")):
        img = cv2.imread(str(path))
        m = cv2.imread(str(DATA_DIR / "masks" / path.name))
        m = cv2.resize(m, img.shape[1::-1], interpolation=cv2.INTER_NEAREST)
        benign = (m[..., 0] < 50) & (m[..., 1] > 200) & (m[..., 2] < 50)
        malignant = (m[..., 0] < 50) & (m[..., 1] < 50) & (m[..., 2] > 200)
        if not (benign.any() or malignant.any()):
            label = 2
        else:
            label = 0 if benign.sum() >= malignant.sum() else 1
        yield img, benign | malignant, label, re.split(r"[_\-]", path.name)[0]

In [ ]:
reader = read_busi if DATASET == "BUSI" else read_uclm
rgb, gray, masks, y, groups = [], [], [], [], []
for img, mask, label, group in reader():
    rgb.append(cv2.resize(img[..., ::-1], (SIZE, SIZE)))
    gray.append(cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (SIZE, SIZE)))
    masks.append(cv2.resize(mask.astype(np.uint8) * 255, (SIZE, SIZE), interpolation=cv2.INTER_NEAREST))
    y.append(label)
    groups.append(group)

rgb, gray, masks, y = np.stack(rgb), np.stack(gray), np.stack(masks), np.array(y)
groups = np.unique(groups, return_inverse=True)[1]
folds = list(StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED).split(y, y, groups))
joblib.dump(dict(y=y, groups=groups, folds=folds), OUT_DIR / "folds.pkl")

## Input conditions

Masks are fused with the grayscale image by a single-level Haar wavelet rule.
The shuffled condition gives each image the expert mask of another image of the same class
within the same split, so the class cue is kept but the lesion location is not.

In [ ]:
def haar_fusion(image, mask):
    a1, d1 = pywt.dwt2(image.astype(float), "haar")
    a2, d2 = pywt.dwt2(mask.astype(float), "haar")
    details = tuple(np.where(np.abs(u) > np.abs(v), u, v) for u, v in zip(d1, d2))
    fused = np.clip(pywt.idwt2(((a1 + a2) / 2, details), "haar"), 0, 255).astype(np.uint8)
    return np.repeat(fused[..., None], 3, axis=2)


def shuffled_masks(k):
    rng = np.random.default_rng(SEED + k)
    out = masks.copy()
    for split in folds[k]:
        for c in range(3):
            idx = split[y[split] == c]
            out[idx] = masks[rng.permutation(idx)]
    return out


def condition_images(condition, k):
    if condition == "raw":
        return rgb
    if condition == "oracle":
        m = masks
    elif condition == "predicted":
        m = predicted[k]
    else:
        m = shuffled_masks(k)
    return np.stack([haar_fusion(g, mm) for g, mm in zip(gray, m)])

## Predicted masks

A U-Net with a MobileNetV2 encoder is trained per outer fold. Training images receive
out-of-fold masks from two patient-disjoint halves; test images receive the average of both models.

In [ ]:
def build_unet():
    layers = tf.keras.layers
    base = tf.keras.applications.MobileNetV2((SIZE, SIZE, 3), include_top=False, weights="imagenet")
    names = ["block_1_expand_relu", "block_3_expand_relu", "block_6_expand_relu",
             "block_13_expand_relu", "block_16_project"]
    skips = [base.get_layer(n).output for n in names]
    x = skips[-1]
    for skip, filters in zip(reversed(skips[:-1]), [256, 128, 64, 32]):
        x = layers.Conv2DTranspose(filters, 3, 2, "same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(layers.Concatenate()([x, skip]))
    x = layers.Conv2DTranspose(16, 3, 2, "same", activation="relu")(x)
    return tf.keras.Model(base.input, layers.Conv2D(1, 1, activation="sigmoid")(x))


def bce_dice_loss(t, p):
    inter = tf.reduce_sum(t * p, [1, 2, 3])
    union = tf.reduce_sum(t + p, [1, 2, 3])
    dice = tf.reduce_mean((2 * inter + 1) / (union + 1))
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(t, p)) + 1 - dice


def predict_masks(k, epochs=40):
    tr, te = folds[k]
    X = tf.keras.applications.mobilenet_v2.preprocess_input(
        np.repeat(gray[..., None], 3, axis=3).astype("float32"))
    M = (masks[..., None] > 0).astype("float32")
    prob = np.zeros(gray.shape, np.float32)
    patients = np.random.default_rng(SEED + k).permutation(np.unique(groups[tr]))
    halves = [tr[np.isin(groups[tr], h)] for h in np.array_split(patients, 2)]
    for j in range(2):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED + 10 * k + j)
        model = build_unet()
        model.compile(tf.keras.optimizers.Adam(3e-4), bce_dice_loss)
        model.fit(X[halves[1 - j]], M[halves[1 - j]], batch_size=16, epochs=epochs, verbose=0)
        prob[halves[j]] = model.predict(X[halves[j]], verbose=0)[..., 0]
        prob[te] += model.predict(X[te], verbose=0)[..., 0] / 2
    return (prob >= 0.5).astype(np.uint8) * 255


predicted = [predict_masks(k) for k in range(N_FOLDS)]
joblib.dump(dict(expert=masks, predicted=predicted), OUT_DIR / "masks.pkl")

## Features

MobileNetV1 is frozen, so its activations are computed once for each image and for five
augmented copies. A Dense-512 head is trained per fold and condition, and its Dense-512
activations are the features used by the GA.

In [ ]:
backbone = tf.keras.applications.MobileNet(weights="imagenet", include_top=False, input_shape=(SIZE, SIZE, 3))


def augment(img, rng):
    angle, zoom = rng.uniform(-20, 20), rng.uniform(0.9, 1.1)
    m = cv2.getRotationMatrix2D((SIZE / 2, SIZE / 2), angle, zoom)
    m[0, 1] += rng.uniform(-0.1, 0.1)
    m[:, 2] += rng.uniform(-0.15, 0.15, 2) * SIZE
    out = cv2.warpAffine(img, m, (SIZE, SIZE), borderMode=cv2.BORDER_REPLICATE)
    return out[:, ::-1].copy() if rng.random() < 0.5 else out


def activations(images):
    views = []
    for v in range(1 + N_AUG):
        imgs = images if v == 0 else np.stack(
            [augment(im, np.random.default_rng([SEED, i, v])) for i, im in enumerate(images)])
        batches = [backbone(tf.keras.applications.mobilenet.preprocess_input(imgs[s:s + 64].astype("float32")),
                            training=False).numpy() for s in range(0, len(imgs), 64)]
        views.append(np.concatenate(batches).reshape(len(imgs), -1).astype("float16"))
    return np.stack(views)


def train_head(acts, train, val, seed):
    layers, l2 = tf.keras.layers, tf.keras.regularizers.l2(0.01)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    inp = layers.Input((acts.shape[2],))
    x = layers.Dense(512, "relu", kernel_regularizer=l2, name="f512")(inp)
    x = layers.BatchNormalization()(layers.Dropout(0.3)(x))
    x = layers.Dense(120, "relu", kernel_regularizer=l2)(x)
    x = layers.BatchNormalization()(layers.Dropout(0.2)(x))
    model = tf.keras.Model(inp, layers.Dense(3, "softmax")(x))
    model.compile(tf.keras.optimizers.Adam(1e-3), "categorical_crossentropy")
    target = np.eye(3)[y]
    rng = np.random.default_rng(seed)
    best, best_weights, wait = np.inf, None, 0
    for _ in range(200):
        view = rng.integers(1, acts.shape[0], len(train))
        model.fit(acts[view, train].astype("float32"), target[train], batch_size=32, epochs=1, verbose=0)
        loss = model.evaluate(acts[0, val].astype("float32"), target[val], verbose=0)
        if loss < best:
            best, best_weights, wait = loss, model.get_weights(), 0
        else:
            wait += 1
            if wait >= 20:
                break
    model.set_weights(best_weights)
    return tf.keras.Model(model.input, model.get_layer("f512").output)

In [ ]:
for condition in CONDITIONS:
    for k, (tr, te) in enumerate(folds):
        acts = activations(condition_images(condition, k))
        order = np.random.default_rng(SEED + k).permutation(tr)
        n_val = len(tr) // 5
        head = train_head(acts, order[n_val:], order[:n_val], SEED + k)
        X = head.predict(acts[0].astype("float32"), verbose=0)
        np.savez_compressed(OUT_DIR / "features" / f"{condition}_fold{k}.npz", X=X)